In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [36]:
!pip install timm --quiet
!pip install rasterio --quiet
!pip install scikit-image --quiet
from skimage.transform import resize

In [41]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================
import os, random, warnings, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingLR
import torchvision.models as models
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import timm
from torch.utils.data import WeightedRandomSampler
print("✓ imported")
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}, timm: {timm.__version__}")

✓ imported
Device: cuda
PyTorch: 2.8.0+cu126, timm: 1.0.20


In [22]:
# CELL 0: PROBE — run this first before anything else
import os

ROOT = "/kaggle/input/beyond-visible-spectrum-ai-for-agriculture-2026p2"

def probe_dir(path, depth=0, max_depth=3, max_items=5):
    if depth > max_depth: return
    if not os.path.exists(path):
        print("  " * depth + f"❌ MISSING: {path}")
        return
    items = sorted(os.listdir(path))
    print("  " * depth + f"📁 {os.path.basename(path)}/ ({len(items)} items)")
    for item in items[:max_items]:
        full = os.path.join(path, item)
        if os.path.isdir(full):
            probe_dir(full, depth+1, max_depth, max_items)
        else:
            print("  " * (depth+1) + f"📄 {item}")
    if len(items) > max_items:
        print("  " * (depth+1) + f"... and {len(items)-max_items} more")

probe_dir(ROOT, max_depth=4)

📁 beyond-visible-spectrum-ai-for-agriculture-2026p2/ (2 items)
  📁 ICPR02/ (1 items)
    📁 kaggle/ (5 items)
      📁 Aphid/ (290 items)
        📁 0041231a3f6f4fa9b07a04234cef4627/ (12 items)
          📄 B1.tif
          📄 B11.tif
          📄 B12.tif
          📄 B2.tif
          📄 B3.tif
          ... and 7 more
        📁 00e6adf1215344a0aa3d396aa50eff0c/ (12 items)
          📄 B1.tif
          📄 B11.tif
          📄 B12.tif
          📄 B2.tif
          📄 B3.tif
          ... and 7 more
        📁 018a98b4251441f0b9e73b9d286541f2/ (12 items)
          📄 B1.tif
          📄 B11.tif
          📄 B12.tif
          📄 B2.tif
          📄 B3.tif
          ... and 7 more
        📁 0299e35c64b74d3d896f7a22227cde31/ (12 items)
          📄 B1.tif
          📄 B11.tif
          📄 B12.tif
          📄 B2.tif
          📄 B3.tif
          ... and 7 more
        📁 035d3057f7af4f1c9b47e2a325b71be2/ (12 items)
          📄 B1.tif
          📄 B11.tif
          📄 B12.tif
          📄 B2.tif
          📄 B3.tif
    

In [23]:
# ============================================================
# CELL 2: EXACT PATHS (verified from probe output)
# ============================================================
ROOT     = "/kaggle/input/beyond-visible-spectrum-ai-for-agriculture-2026p2"
ICPR_DIR = os.path.join(ROOT, "ICPR02", "kaggle")
EVAL_DIR = os.path.join(ICPR_DIR, "evaluation")

# Labelled classes — exact folder names as seen in probe
CLASS_MAP   = {"Aphid": 0, "Blast": 1, "RPH": 2, "Rust": 3}
CLASS_NAMES = ["Aphid", "Blast", "RPH", "Rust"]

# SSL unlabelled — archive/share/train/* folders contain sample dirs directly
SSL_ROOTS = [
    os.path.join(ROOT, "archive", "share", "train", "rice"),
    os.path.join(ROOT, "archive", "share", "train", "maize"),
    os.path.join(ROOT, "archive", "share", "train", "soybean"),   # note: "soybean" not "soyabean"
    os.path.join(ROOT, "archive", "share", "train", "wheat"),
    os.path.join(ROOT, "archive", "share", "val"),                # numbered folders directly
]

BANDS = [
    "B1.tif","B2.tif","B3.tif","B4.tif",
    "B5.tif","B6.tif","B7.tif",
    "B8.tif","B8A.tif","B9.tif",
    "B11.tif","B12.tif"
]
IN_CH     = 12
CROP_SIZE = 64

# ---- Quick sanity check ----
print("\n--- Labelled ---")
for cls in CLASS_MAP:
    p = os.path.join(ICPR_DIR, cls)
    n = len(os.listdir(p)) if os.path.exists(p) else "❌ MISSING"
    print(f"  {cls}: {n}")
print(f"  evaluation: {len(os.listdir(EVAL_DIR)) if os.path.exists(EVAL_DIR) else '❌ MISSING'}")

print("\n--- SSL unlabelled ---")
for d in SSL_ROOTS:
    n = len(os.listdir(d)) if os.path.exists(d) else "❌ MISSING"
    print(f"  {os.path.relpath(d, ROOT)}: {n}")



--- Labelled ---
  Aphid: 290
  Blast: 75
  RPH: 495
  Rust: 40
  evaluation: 40

--- SSL unlabelled ---
  archive/share/train/rice: 196
  archive/share/train/maize: 163
  archive/share/train/soybean: 233
  archive/share/train/wheat: 140
  archive/share/val: 432


In [24]:
# ============================================================
# CELL 3: DATA COLLECTION FUNCTIONS
# ============================================================
def is_valid_sample(folder):
    """Check folder has at least B4.tif (reference band)."""
    return os.path.isdir(folder) and os.path.exists(os.path.join(folder, "B4.tif"))


def collect_labelled():
    paths, labels = [], []
    for cls, idx in CLASS_MAP.items():
        d = os.path.join(ICPR_DIR, cls)
        for f in sorted(os.listdir(d)):
            full = os.path.join(d, f)
            if is_valid_sample(full):
                paths.append(full)
                labels.append(idx)
    print(f"Labelled samples: {len(paths)}")
    for cls, idx in CLASS_MAP.items():
        print(f"  {cls}: {labels.count(idx)}")
    return paths, labels


def collect_ssl():
    """
    SSL structure:
      train/rice|maize|soybean|wheat/<sample_folder>/B*.tif
      val/<sample_folder>/B*.tif
    """
    paths = []
    for root in SSL_ROOTS:
        if not os.path.exists(root):
            print(f"  ⚠️  SSL path missing: {root}")
            continue
        for f in os.listdir(root):
            full = os.path.join(root, f)
            if is_valid_sample(full):
                paths.append(full)
    print(f"SSL (unlabelled) samples: {len(paths)}")
    return paths


def collect_eval():
    paths, ids = [], []
    if not os.path.exists(EVAL_DIR):
        print("❌ Evaluation dir missing")
        return paths, ids
    for f in sorted(os.listdir(EVAL_DIR)):
        full = os.path.join(EVAL_DIR, f)
        if is_valid_sample(full):
            paths.append(full)
            ids.append(f)
    print(f"Evaluation samples: {len(paths)}")
    return paths, ids


all_paths, all_labels = collect_labelled()
ssl_paths             = collect_ssl()
eval_paths, eval_ids  = collect_eval()



Labelled samples: 900
  Aphid: 290
  Blast: 75
  RPH: 495
  Rust: 40
SSL (unlabelled) samples: 432
Evaluation samples: 40


In [37]:
# ============================================================
# CELL 4 (REPLACE THIS ENTIRELY): FIXED load_stack
# ============================================================
!pip install scikit-image --quiet
from skimage.transform import resize as sk_resize

def load_stack(folder):
    ref_path = os.path.join(folder, "B4.tif")
    with rasterio.open(ref_path) as ref:
        ref_shape = ref.shape
        ref_tf    = ref.transform
        ref_crs   = ref.crs

    stack = []
    for b in BANDS:
        fp = os.path.join(folder, b)

        # Missing band → zeros
        if not os.path.exists(fp):
            stack.append(np.zeros(ref_shape, dtype=np.float32))
            continue

        with rasterio.open(fp) as src:
            band = src.read(1).astype(np.float32)

            if src.shape != ref_shape:
                # Has valid CRS on both sides → proper reproject
                if src.crs is not None and ref_crs is not None:
                    out = np.zeros(ref_shape, dtype=np.float32)
                    reproject(band, out,
                              src_transform=src.transform, src_crs=src.crs,
                              dst_transform=ref_tf,        dst_crs=ref_crs,
                              resampling=Resampling.bilinear)
                    band = out
                else:
                    # Missing CRS (common in archive/share data) → simple resize
                    band = sk_resize(band, ref_shape,
                                     order=1,
                                     preserve_range=True,
                                     anti_aliasing=True).astype(np.float32)

            stack.append(band)

    return np.stack(stack, axis=0)  # (12, H, W)


print("✓ load_stack updated — re-run Cell 11 (pretrain) now")

✓ load_stack updated — re-run Cell 11 (pretrain) now


In [26]:

# ============================================================
# CELL 5: COMPUTE BAND STATISTICS
# ============================================================
def compute_stats(paths, n=400):
    sample = random.sample(paths, min(n, len(paths)))
    print(f"Computing stats from {len(sample)} samples...")
    sums = np.zeros(12, dtype=np.float64)
    sq   = np.zeros(12, dtype=np.float64)
    cnt  = np.zeros(12, dtype=np.float64)
    for i, p in enumerate(sample):
        if i % 100 == 0: print(f"  {i}/{len(sample)}")
        try:
            s = load_stack(p)
            for b in range(12):
                v = s[b].ravel()
                sums[b] += v.sum()
                sq[b]   += (v**2).sum()
                cnt[b]  += len(v)
        except: continue
    means = (sums / cnt).astype(np.float32)
    stds  = np.sqrt(np.maximum(sq/cnt - means**2, 1e-6)).astype(np.float32)
    print("BAND_MEANS =", np.round(means, 1).tolist())
    print("BAND_STDS  =", np.round(stds,  1).tolist())
    return means, stds

# Use all labelled + 300 random SSL samples for robust stats
stat_pool = all_paths + random.sample(ssl_paths, min(300, len(ssl_paths)))
BAND_MEANS, BAND_STDS = compute_stats(stat_pool, n=500)



Computing stats from 500 samples...
  0/500
  100/500
  200/500
  300/500
  400/500
BAND_MEANS = [2417.800048828125, 2521.60009765625, 2685.300048828125, 2709.300048828125, 3137.699951171875, 3651.300048828125, 3864.89990234375, 3888.300048828125, 4015.800048828125, 4821.39990234375, 3413.60009765625, 2697.0]
BAND_STDS  = [3222.800048828125, 3094.0, 2726.89990234375, 2554.699951171875, 2555.5, 2272.0, 2153.60009765625, 2140.699951171875, 2065.60009765625, 3682.699951171875, 1580.9000244140625, 1418.5]


In [27]:
# ============================================================
# CELL 6: TRANSFORMS
# ============================================================
def normalize(x):
    x = (x - BAND_MEANS[:,None,None]) / (BAND_STDS[:,None,None] + 1e-6)
    return np.clip(x, -3., 3.).astype(np.float32)


def center_crop_or_pad(x, size=CROP_SIZE):
    _, h, w = x.shape
    if h < size or w < size:
        x = np.pad(x, ((0,0),(0,max(0,size-h)),(0,max(0,size-w))), mode='reflect')
        _, h, w = x.shape
    r = (h-size)//2; c = (w-size)//2
    return x[:, r:r+size, c:c+size]


def random_crop(x, size=CROP_SIZE):
    _, h, w = x.shape
    if h < size or w < size:
        x = np.pad(x, ((0,0),(0,max(0,size-h)),(0,max(0,size-w))), mode='reflect')
        _, h, w = x.shape
    r = random.randint(0, h-size); c = random.randint(0, w-size)
    return x[:, r:r+size, c:c+size]


def aug_supervised(x):
    if random.random() > .5: x = x[:,:,::-1].copy()
    if random.random() > .5: x = x[:,::-1,:].copy()
    k = random.randint(0,3)
    if k: x = np.rot90(x, k, axes=(1,2)).copy()
    if random.random() > .6:
        x = x.copy()
        for b in random.sample(range(12), random.randint(1,2)):
            x[b] = 0.
    if random.random() > .5:
        x = x + np.random.randn(*x.shape).astype(np.float32) * 0.04
    return x


def aug_ssl(x):
    x = random_crop(x)
    if random.random() > .5: x = x[:,:,::-1].copy()
    if random.random() > .5: x = x[:,::-1,:].copy()
    k = random.randint(0,3)
    if k: x = np.rot90(x, k, axes=(1,2)).copy()
    scale = (1 + np.random.randn(12).astype(np.float32)*0.15)[:,None,None]
    x = x * scale
    if random.random() > .5:
        x = x.copy()
        for b in random.sample(range(12), random.randint(1,3)):
            x[b] = 0.
    return x + np.random.randn(*x.shape).astype(np.float32) * 0.05


In [28]:

# ============================================================
# CELL 7: DATASETS
# ============================================================
class LabelledDS(Dataset):
    def __init__(self, paths, labels, train=True):
        self.paths = paths; self.labels = labels; self.train = train

    def __len__(self): return len(self.paths)

    def __getitem__(self, i):
        x = load_stack(self.paths[i])
        x = center_crop_or_pad(x)
        if self.train: x = aug_supervised(x)
        return torch.from_numpy(normalize(x).copy()).float(), self.labels[i]


class SSLDS(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self): return len(self.paths)

    def __getitem__(self, i):
        x  = load_stack(self.paths[i])
        v1 = normalize(aug_ssl(x))
        v2 = normalize(aug_ssl(x))
        return torch.from_numpy(v1.copy()).float(), torch.from_numpy(v2.copy()).float()


class EvalDS(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self): return len(self.paths)

    def __getitem__(self, i):
        x = load_stack(self.paths[i])
        x = center_crop_or_pad(x)
        return torch.from_numpy(normalize(x).copy()).float()


In [29]:

# ============================================================
# CELL 8: CLASS-BALANCED SAMPLER
# ============================================================
def make_balanced_sampler(labels):
    """
    Critical for this dataset: Rust=40, Blast=75 vs RPH=495.
    Without balancing, model ignores minority classes.
    """
    counts  = np.bincount(labels)
    weights = 1.0 / counts[labels]
    return WeightedRandomSampler(
        weights=torch.from_numpy(weights).float(),
        num_samples=len(labels),
        replacement=True
    )


In [30]:

# ============================================================
# CELL 9: MODEL
# ============================================================
class Encoder(nn.Module):
    def __init__(self, in_ch=12, pretrained=True):
        super().__init__()
        self.net = timm.create_model(
            'efficientnet_b0', pretrained=pretrained,
            num_classes=0, global_pool='avg', in_chans=in_ch
        )
        self.embed_dim = self.net.num_features  # 1280

    def forward(self, x): return self.net(x)


class ProjHead(nn.Module):
    """SimCLR projection — discarded after pretraining."""
    def __init__(self, dim=1280, out=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.ReLU(),
            nn.Linear(dim, out)
        )
    def forward(self, x): return F.normalize(self.net(x), dim=-1)


class ClsHead(nn.Module):
    def __init__(self, dim=1280, n_cls=4, drop=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Dropout(drop/2),
            nn.Linear(256, n_cls)
        )
    def forward(self, x): return self.net(x)


In [31]:
# ============================================================
# CELL 10: NT-XENT LOSS
# ============================================================
class NTXent(nn.Module):
    def __init__(self, temp=0.5):
        super().__init__(); self.temp = temp

    def forward(self, z1, z2):
        N = z1.shape[0]
        z = torch.cat([z1, z2])
        sim = torch.mm(z, z.T) / self.temp
        sim.fill_diagonal_(-1e9)
        labels = torch.cat([torch.arange(N,2*N), torch.arange(0,N)]).to(z.device)
        return F.cross_entropy(sim, labels)


In [38]:
# ============================================================
# CELL 11: SSL PRETRAINING
# ============================================================
def pretrain(ssl_paths, epochs=25, bs=32):
    if not ssl_paths:
        print("No SSL data — skipping."); return None

    print(f"\n{'='*55}")
    print(f" SimCLR PRETRAINING  |  {len(ssl_paths)} unlabelled samples")
    print(f"{'='*55}")

    loader  = DataLoader(SSLDS(ssl_paths), batch_size=bs, shuffle=True,
                         num_workers=2, pin_memory=True, drop_last=True)
    enc     = Encoder(pretrained=True).to(device)
    proj    = ProjHead(enc.embed_dim).to(device)
    loss_fn = NTXent()
    opt     = optim.AdamW(list(enc.parameters())+list(proj.parameters()),
                          lr=1e-3, weight_decay=1e-4)
    sch     = CosineAnnealingLR(opt, T_max=epochs)
    best    = 1e9

    for ep in range(epochs):
        enc.train(); proj.train()
        total = 0
        for v1, v2 in loader:
            v1, v2 = v1.to(device), v2.to(device)
            loss = loss_fn(proj(enc(v1)), proj(enc(v2)))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(enc.parameters())+list(proj.parameters()), 1.0)
            opt.step(); total += loss.item()
        sch.step()
        avg = total / len(loader)
        if avg < best:
            best = avg
            torch.save(enc.state_dict(), "/kaggle/working/ssl_encoder.pth")
        print(f"  Ep {ep+1:02d}/{epochs}  loss={avg:.4f}{'  ✓' if avg==best else ''}")

    enc.load_state_dict(torch.load("/kaggle/working/ssl_encoder.pth"))
    print(f"Pretraining done. Best loss: {best:.4f}")
    return enc


pretrained_enc = pretrain(ssl_paths, epochs=25, bs=32)



 SimCLR PRETRAINING  |  432 unlabelled samples
  Ep 01/25  loss=3.9088  ✓
  Ep 02/25  loss=3.7449  ✓
  Ep 03/25  loss=3.6701  ✓
  Ep 04/25  loss=3.5632  ✓
  Ep 05/25  loss=3.5513  ✓
  Ep 06/25  loss=3.4915  ✓
  Ep 07/25  loss=3.4591  ✓
  Ep 08/25  loss=3.5079
  Ep 09/25  loss=3.4630
  Ep 10/25  loss=3.4137  ✓
  Ep 11/25  loss=3.3707  ✓
  Ep 12/25  loss=3.3790
  Ep 13/25  loss=3.2747  ✓
  Ep 14/25  loss=3.3062
  Ep 15/25  loss=3.2463  ✓
  Ep 16/25  loss=3.2484
  Ep 17/25  loss=3.2430  ✓
  Ep 18/25  loss=3.1955  ✓
  Ep 19/25  loss=3.2078
  Ep 20/25  loss=3.1657  ✓
  Ep 21/25  loss=3.1828
  Ep 22/25  loss=3.1812
  Ep 23/25  loss=3.0988  ✓
  Ep 24/25  loss=3.1115
  Ep 25/25  loss=3.2166
Pretraining done. Best loss: 3.0988


In [43]:
# ============================================================
# CELL 12: SUPERVISED FINE-TUNING
# ============================================================
train_p, val_p, train_l, val_l = train_test_split(
    all_paths, all_labels,
    test_size=0.2, stratify=all_labels, random_state=42
)
print(f"Train: {len(train_p)}  Val: {len(val_p)}")
for cls, idx in CLASS_MAP.items():
    print(f"  {cls} — train: {train_l.count(idx)}  val: {val_l.count(idx)}")

sampler    = make_balanced_sampler(train_l)
tr_loader  = DataLoader(LabelledDS(train_p, train_l, train=True),
                        batch_size=16, sampler=sampler,
                        num_workers=2, pin_memory=True)
vl_loader  = DataLoader(LabelledDS(val_p, val_l, train=False),
                        batch_size=16, shuffle=False,
                        num_workers=2, pin_memory=True)

# Class weights for loss (inverse frequency) — extra boost for minority classes
counts      = np.bincount(all_labels)
cls_weights = torch.tensor(1.0 / counts, dtype=torch.float32).to(device)
cls_weights = cls_weights / cls_weights.sum() * len(CLASS_MAP)
print(f"\nClass weights: {cls_weights.cpu().numpy().round(3)}")


def finetune(pretrained_enc=None, epochs=60):
    print(f"\n{'='*55}")
    print(f" SUPERVISED FINE-TUNING  |  {epochs} epochs")
    print(f"{'='*55}")

    enc = pretrained_enc if pretrained_enc is not None \
          else Encoder(pretrained=True).to(device)
    cls = ClsHead(enc.embed_dim).to(device)

    # Weighted CE + label smoothing for imbalanced classes
    crit = nn.CrossEntropyLoss(weight=cls_weights, label_smoothing=0.1)
    opt  = optim.AdamW([
        {"params": enc.parameters(), "lr": 5e-5},
        {"params": cls.parameters(), "lr": 3e-4},
    ], weight_decay=1e-4)
    sch  = OneCycleLR(opt, max_lr=[2e-4, 1e-3],
                      steps_per_epoch=len(tr_loader),
                      epochs=epochs, pct_start=0.1)

    best_acc = 0.; patience = 0; PATIENCE = 15

    for ep in range(epochs):
        enc.train(); cls.train()
        corr = tot = 0; loss_sum = 0.
        for x, y in tr_loader:
            x, y = x.to(device), y.to(device)
            out  = cls(enc(x))
            loss = crit(out, y)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(enc.parameters())+list(cls.parameters()), 1.0)
            opt.step(); sch.step()
            loss_sum += loss.item()
            corr     += (out.argmax(1)==y).sum().item()
            tot      += y.size(0)

        tr_acc = corr / tot
        enc.eval(); cls.eval()
        corr = tot = 0
        with torch.no_grad():
            for x, y in vl_loader:
                x, y = x.to(device), y.to(device)
                corr += (cls(enc(x)).argmax(1)==y).sum().item()
                tot  += y.size(0)
        vl_acc = corr / tot

        if vl_acc > best_acc:
            best_acc = vl_acc; patience = 0
            torch.save({"enc": enc.state_dict(), "cls": cls.state_dict()},
                       "/kaggle/working/best_model.pth")
            tag = "✓"
        else:
            patience += 1
            tag = f"patience {patience}/{PATIENCE}"

        print(f"Ep {ep+1:02d}  loss={loss_sum/len(tr_loader):.4f}  "
              f"tr={tr_acc:.3f}  vl={vl_acc:.3f}  best={best_acc:.3f}  {tag}")

        if patience >= PATIENCE:
            print("Early stopping."); break

    return enc, cls


enc, cls = finetune(pretrained_enc, epochs=60)



Train: 720  Val: 180
  Aphid — train: 232  val: 58
  Blast — train: 60  val: 15
  RPH — train: 396  val: 99
  Rust — train: 32  val: 8

Class weights: [0.315 1.218 0.184 2.283]

 SUPERVISED FINE-TUNING  |  60 epochs
Ep 01  loss=1.3027  tr=0.311  vl=0.306  best=0.306  ✓
Ep 02  loss=1.0235  tr=0.390  vl=0.161  best=0.306  patience 1/15
Ep 03  loss=0.8263  tr=0.458  vl=0.172  best=0.306  patience 2/15
Ep 04  loss=0.7374  tr=0.475  vl=0.178  best=0.306  patience 3/15
Ep 05  loss=0.7293  tr=0.490  vl=0.194  best=0.306  patience 4/15
Ep 06  loss=0.6390  tr=0.542  vl=0.183  best=0.306  patience 5/15
Ep 07  loss=0.6614  tr=0.589  vl=0.244  best=0.306  patience 6/15
Ep 08  loss=0.6324  tr=0.557  vl=0.228  best=0.306  patience 7/15
Ep 09  loss=0.5645  tr=0.610  vl=0.278  best=0.306  patience 8/15
Ep 10  loss=0.6106  tr=0.594  vl=0.350  best=0.350  ✓
Ep 11  loss=0.5341  tr=0.675  vl=0.328  best=0.350  patience 1/15
Ep 12  loss=0.5245  tr=0.625  vl=0.422  best=0.422  ✓
Ep 13  loss=0.5444  tr=0.678

In [44]:
# ============================================================
# CELL 13: EVALUATION REPORT
# ============================================================
ckpt = torch.load("/kaggle/working/best_model.pth")
enc.load_state_dict(ckpt["enc"])
cls.load_state_dict(ckpt["cls"])

enc.eval(); cls.eval()
preds, gts = [], []
with torch.no_grad():
    for x, y in vl_loader:
        preds.extend(cls(enc(x.to(device))).argmax(1).cpu().tolist())
        gts.extend(y.tolist())

print(f"\nVal Accuracy: {sum(p==g for p,g in zip(preds,gts))/len(gts):.4f}")
print(classification_report(gts, preds, target_names=CLASS_NAMES))




Val Accuracy: 0.7111
              precision    recall  f1-score   support

       Aphid       0.88      0.90      0.89        58
       Blast       0.17      0.33      0.22        15
         RPH       0.86      0.67      0.75        99
        Rust       0.36      0.62      0.45         8

    accuracy                           0.71       180
   macro avg       0.57      0.63      0.58       180
weighted avg       0.79      0.71      0.74       180



In [47]:
# ============================================================
# CELL 14: SUBMISSION
# ============================================================
print(f"\nGenerating submission for {len(eval_paths)} samples...")

ev_loader = DataLoader(EvalDS(eval_paths), batch_size=16,
                       shuffle=False, num_workers=2)
enc.eval(); cls.eval()
test_preds = []
with torch.no_grad():
    for x in ev_loader:
        test_preds.extend(cls(enc(x.to(device))).argmax(1).cpu().tolist())

inv = {v: k for k, v in CLASS_MAP.items()}
sub = pd.DataFrame({
    "ID":    eval_ids,
    "Category": [inv[p] for p in test_preds]
})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print("✓ submission.csv saved!")
print(sub.head(10))
print("\nDistribution:")
print(sub["Category"].value_counts())


Generating submission for 40 samples...
✓ submission.csv saved!
                                 ID Category
0  05835a9764364429b5ac3e11b052649d     Rust
1  13739e32e7a84f669e6ef1284715e93b    Aphid
2  1a419acc1ecc467897d5477a47353fa8      RPH
3  2fb5f497ae1b4b1eb7e8d7ced143aa46    Aphid
4  310283f25b5f4b038114acbb6d61a357     Rust
5  342a74c6432d458d956ba20ad71339c0     Rust
6  34ff69d337a448d5acf24c06d134c07a    Aphid
7  36eb3780202748baa0164000b4e2e5b3      RPH
8  4da1b698cdad4c8db6c1716a51a56bd4    Aphid
9  5bf370118f3043f1bbeafbb91bd78f32      RPH

Distribution:
Category
RPH      18
Aphid    14
Rust      7
Blast     1
Name: count, dtype: int64
